# DNS Tunneling Detection — Colab Demo

**Self-contained** end-to-end notebook. No clone required — all source code from `src/` is embedded below so you can run every cell top-to-bottom on a fresh Colab runtime.

What this notebook does:

1. Install dependencies (XGBoost is the only thing not preinstalled on Colab).
2. Define all 11 features (`shannon entropy`, lengths, n-grams, character ratios).
3. Load a dataset — either a CIC CSV you upload, or a synthetic stand-in generated on the fly.
4. Train **Random Forest** and **XGBoost** with 5-fold cross-validation.
5. Evaluate on a held-out test split: metrics table, confusion matrices, ROC curves, feature importances.

**Repo**: https://github.com/df-DNS-Tunneling-Detection/DNS_Tunneling_Detection

**Open this notebook in Colab**: [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/df-DNS-Tunneling-Detection/DNS_Tunneling_Detection/blob/main/notebooks/colab_demo.ipynb)

## 1. Install dependencies

In [ ]:
!pip install -q xgboost seaborn joblib

In [ ]:
import math
import random
import string
from collections import Counter
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from xgboost import XGBClassifier

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (7, 4)

## 2. Feature extraction (embedded `src/features.py`)

Eleven metadata features per DNS query. Shannon entropy is computed as $H(X) = -\sum_i p_i \log_2 p_i$ over the empirical character distribution.

In [ ]:
VOWELS = set("aeiouAEIOU")

def shannon_entropy(s):
    if not s: return 0.0
    counts = Counter(s); n = len(s)
    return -sum((c/n) * math.log2(c/n) for c in counts.values())

def query_length(s): return len(s) if s else 0

def subdomain_length(s):
    if not s: return 0
    parts = s.split(".")
    if len(parts) <= 2: return 0
    return sum(len(p) for p in parts[:-2]) + max(0, len(parts) - 3)

def digit_ratio(s):
    if not s: return 0.0
    return sum(ch.isdigit() for ch in s) / len(s)

def uppercase_ratio(s):
    if not s: return 0.0
    return sum(ch.isupper() for ch in s) / len(s)

def unique_char_count(s): return len(set(s)) if s else 0

def vowel_consonant_ratio(s):
    if not s: return 0.0
    letters = [ch for ch in s if ch.isalpha()]
    if not letters: return 0.0
    v = sum(ch in VOWELS for ch in letters)
    c = len(letters) - v
    return float(v) if c == 0 else v / c

def label_count(s): return 0 if not s else s.count(".") + 1

def longest_label_length(s):
    if not s: return 0
    return max(len(p) for p in s.split("."))

def bigram_entropy(s):
    if not s or len(s) < 2: return 0.0
    bigrams = [s[i:i+2] for i in range(len(s) - 1)]
    counts = Counter(bigrams); n = len(bigrams)
    return -sum((c/n) * math.log2(c/n) for c in counts.values())

def non_alphanum_ratio(s):
    if not s: return 0.0
    return sum(not (ch.isalnum() or ch == ".") for ch in s) / len(s)

FEATURE_FUNCS = {
    "query_length": query_length,
    "subdomain_length": subdomain_length,
    "entropy": shannon_entropy,
    "bigram_entropy": bigram_entropy,
    "digit_ratio": digit_ratio,
    "uppercase_ratio": uppercase_ratio,
    "unique_char_count": unique_char_count,
    "vowel_consonant_ratio": vowel_consonant_ratio,
    "label_count": label_count,
    "longest_label_length": longest_label_length,
    "non_alphanum_ratio": non_alphanum_ratio,
}

def extract_features(queries):
    queries = queries.fillna("").astype(str)
    return pd.DataFrame(
        {name: queries.map(func) for name, func in FEATURE_FUNCS.items()},
        index=queries.index,
    )

# Sanity check
samples = pd.Series([
    "mail.google.com",
    "aXk1bG9yZW0gaXBzdW0gZG9sb3Igc2l0.tunnel.evil.com",
])
extract_features(samples)

## 3. Dataset loader (embedded `src/preprocess.py`)

Works on any CSV with a query/text column + binary label column.

In [ ]:
_QUERY_CANDIDATES = ("query", "domain", "name", "qname", "Domain", "Query")
_LABEL_CANDIDATES = ("label", "Label", "class", "Class", "is_malicious", "Type")
_POSITIVE_TOKENS = {"1", "malicious", "malware", "tunneling", "tunnel",
                    "doh-malicious", "doh_malicious", "attack", "evil", "true"}

def _find_column(df, candidates):
    for name in candidates:
        if name in df.columns: return name
    return None

def _normalize_labels(series):
    if pd.api.types.is_numeric_dtype(series):
        return (series > 0).astype(int)
    lower = series.astype(str).str.strip().str.lower()
    return lower.isin(_POSITIVE_TOKENS).astype(int)

def load_dataset(path, query_col=None, label_col=None):
    path = Path(path)
    if path.is_dir():
        frames = [pd.read_csv(p) for p in sorted(path.glob("*.csv"))]
        df = pd.concat(frames, ignore_index=True)
    else:
        df = pd.read_csv(path)
    df = df.dropna(how="all").reset_index(drop=True)

    label_col = label_col or _find_column(df, _LABEL_CANDIDATES)
    if label_col is None:
        raise ValueError(f"No label column. Columns: {list(df.columns)}")
    labels = _normalize_labels(df[label_col])

    query_col = query_col or _find_column(df, _QUERY_CANDIDATES)
    if query_col is not None:
        data = df[[query_col]].rename(columns={query_col: "query"}).fillna("")
        data["query"] = data["query"].astype(str)
    else:
        data = df.drop(columns=[label_col]).select_dtypes(include="number").copy()
        if data.empty:
            raise ValueError("No query column or numeric features available.")

    keep = ~data.duplicated()
    return data[keep].reset_index(drop=True), labels[keep].reset_index(drop=True)

## 4. Get a dataset

**Option A** — upload a CIC CSV (DoHBrw-2020 or Bell-DNS-2021):
```python
from google.colab import files
uploaded = files.upload()
DATA_PATH = next(iter(uploaded.keys()))
```

**Option B** — generate a synthetic stand-in (used by default below). 2 000 benign + 2 000 base32-style tunneling queries — enough to run the whole pipeline meaningfully.

In [ ]:
BENIGN_TLDS = ["com", "net", "org", "io", "co", "edu", "gov"]
BENIGN_WORDS = ["google", "mail", "drive", "docs", "github", "stackoverflow",
                "amazon", "wikipedia", "twitter", "facebook", "instagram",
                "linkedin", "youtube", "reddit", "medium", "apple", "microsoft",
                "office", "outlook", "azure", "cloud", "api", "cdn", "static",
                "img", "assets", "login", "auth", "search", "news", "blog",
                "shop", "store", "pay", "checkout"]
BENIGN_SUBS = ["", "www", "mail", "api", "cdn", "static", "auth", "m"]

def gen_benign(rng):
    parts = [p for p in (rng.choice(BENIGN_SUBS),
                         rng.choice(BENIGN_WORDS),
                         rng.choice(BENIGN_TLDS)) if p]
    return ".".join(parts)

def gen_tunneling(rng):
    alphabet = string.ascii_lowercase + string.digits
    labels = ["".join(rng.choices(alphabet, k=rng.randint(30, 60)))
              for _ in range(rng.randint(1, 2))]
    domain = (rng.choice(["tunnel", "c2", "exfil", "payload"]) + "." +
              rng.choice(["evil", "attacker", "bad"]) + "." +
              rng.choice(BENIGN_TLDS))
    return ".".join(labels + [domain])

def build_synthetic(n_benign=2000, n_malicious=2000, seed=0):
    rng = random.Random(seed)
    rows = [(gen_benign(rng), 0) for _ in range(n_benign)]
    rows += [(gen_tunneling(rng), 1) for _ in range(n_malicious)]
    rng.shuffle(rows)
    return pd.DataFrame(rows, columns=["query", "label"])

# Generate and save
df = build_synthetic()
DATA_PATH = "sample.csv"
df.to_csv(DATA_PATH, index=False)
print(f"Wrote {len(df)} rows to {DATA_PATH}")
df.sample(6, random_state=1)

## 5. Quick EDA

In [ ]:
data, y = load_dataset(DATA_PATH)
print(f"rows: {len(data):,}   positives: {int(y.sum())}   negatives: {int((1 - y).sum())}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
y.value_counts().rename({0: "benign", 1: "tunneling"}).plot(
    kind="bar", ax=axes[0], color=["#4c72b0", "#dd8452"]
)
axes[0].set_title("Class distribution"); axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=0)

if "query" in data.columns:
    tmp = pd.DataFrame({"length": data["query"].map(query_length), "label": y})
    sns.histplot(data=tmp, x="length", hue="label", bins=40, kde=True,
                 palette="Set1", ax=axes[1])
    axes[1].set_title("Query length by class")
plt.tight_layout(); plt.show()

## 6. Build feature matrix and split

In [ ]:
X = extract_features(data["query"]) if "query" in data.columns else data
print(f"features: {list(X.columns)}")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
X.head()

## 7. Cross-validated training

In [ ]:
models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=200, n_jobs=-1, random_state=42, class_weight="balanced"
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.1,
        eval_metric="logloss", tree_method="hist", n_jobs=-1, random_state=42,
    ),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}
for name, model in models.items():
    print(f"\n[{name}] 5-fold CV:")
    row = {}
    for metric in ("accuracy", "f1", "roc_auc"):
        s = cross_val_score(model, X_train, y_train, cv=cv, scoring=metric, n_jobs=-1)
        row[metric] = float(s.mean())
        print(f"  {metric:10s}: {row[metric]:.4f}  (+/- {s.std():.4f})")
    cv_results[name] = row
pd.DataFrame(cv_results).T.round(4)

## 8. Fit + evaluate on the held-out test split

In [ ]:
test_metrics = {}
roc_data = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    test_metrics[name] = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_prob),
    }
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_data[name] = (fpr, tpr, test_metrics[name]["roc_auc"])
    print(f"\n[{name}]")
    print(classification_report(y_test, y_pred, target_names=["benign", "tunneling"]))

pd.DataFrame(test_metrics).T.round(4)

## 9. Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, (name, model) in zip(axes, models.items()):
    cm = confusion_matrix(y_test, model.predict(X_test))
    ConfusionMatrixDisplay(cm, display_labels=["benign", "tunneling"]).plot(
        ax=ax, colorbar=False, cmap="Blues"
    )
    ax.set_title(name)
plt.tight_layout(); plt.show()

## 10. ROC curves

In [ ]:
plt.figure(figsize=(5, 4.5))
for name, (fpr, tpr, auc) in roc_data.items():
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.4)
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC — model comparison")
plt.legend(loc="lower right"); plt.tight_layout(); plt.show()

## 11. Feature importance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, (name, model) in zip(axes, models.items()):
    importances = model.feature_importances_
    order = np.argsort(importances)[::-1]
    sns.barplot(x=importances[order], y=np.array(X.columns)[order],
                ax=ax, palette="viridis")
    ax.set_title(f"{name} — feature importance"); ax.set_xlabel("")
plt.tight_layout(); plt.show()

## 12. Save trained models

Files end up in the Colab session — use the left sidebar to download them, or mount Drive first.

In [ ]:
joblib.dump(models["Random Forest"], "rf.pkl")
joblib.dump(models["XGBoost"], "xgb.pkl")
print("saved: rf.pkl, xgb.pkl")

## Takeaways

- Length, entropy, and subdomain length dominate feature importance — exactly what intuition predicts.
- RF and XGBoost perform within a fraction of a percent of each other.
- See `reports/report.md` in the repo for the full write-up and limitations.